# Assignment 1 — Gradio dashboard

You're using a notebook for this stage. If you'd rather use a script,
write `app.py` instead and delete this file — `main.py` refuses to run if
it finds both `app.py` and `app.ipynb`, so exactly one of them must exist.

Replace this cell and everything below it with your own code. When run via
`uv run python main.py app`, this notebook must build and launch a Gradio
app (a `demo` that calls `.launch()`) using only what your training stage
already produced — it must never retrain anything itself. At minimum, the
dashboard must let you:

- Compare your three trained models via a prediction-vs-actual plot.
- See feature and/or target distributions.
- For classification: move a decision-threshold slider and watch the
  confusion matrix, and a business-cost number tied to your `REPORT.md`'s
  framing, change with it.

How you structure the code beyond that — cell layout, function
boundaries, naming — is your call to make and be able to defend.

In [ ]:
import matplotlib
matplotlib.use("Agg")                       
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import gradio as gr
from pathlib import Path

ART = Path("artifacts")
preds = pd.read_csv(ART / "test_predictions.csv")
metrics = pd.read_csv(ART / "test_metrics.csv")
features = pd.read_csv(ART / "feature_data.csv")
coefs = pd.read_csv(ART / "coefficients.csv", index_col=0)

MODELS = ["baseline", "scikit-learn", "PyTorch manual loop", "PyTorch nn.Module"]

def plot_pred_vs_actual():
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5), sharex=True, sharey=True)
    lo, hi = preds["actual"].min() - 20, preds["actual"].max() + 20
    for ax, name in zip(axes, MODELS):
        mae = np.abs(preds[name] - preds["actual"]).mean()
        ax.scatter(preds["actual"], preds[name], s=6, alpha=0.4)
        ax.plot([lo, hi], [lo, hi], color="red", linewidth=1, label="perfect prediction")
        ax.set_title(f"{name}\nMAE {mae:.1f} kg")
        ax.set_xlabel("Actual total (kg)")
        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
    axes[0].set_ylabel("Predicted total (kg)")
    axes[0].legend(loc="upper left")
    fig.tight_layout()
    return fig

def plot_distribution(column):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(features[column].dropna(), bins=40)
    ax.set_title(f"Distribution of {column} (all predictable meets)")
    ax.set_xlabel(column)
    ax.set_ylabel("Number of meets")
    fig.tight_layout()
    return fig

def safety_margin(margin, model):
    adjusted = preds[model] - margin                 
    err = adjusted - preds["actual"]
    over = err > 0
    summary = pd.DataFrame({
        "Metric": ["Average error (MAE, kg)",
                   "Lifters overpredicted (%)",
                   "Average overshoot when overpredicted (kg)",
                   "Lifters overpredicted by more than 25 kg (%)"],
        "Value": [round(float(np.abs(err).mean()), 1),
                  round(float(100 * over.mean()), 1),
                  round(float(err[over].mean()), 1) if over.any() else 0.0,
                  round(float(100 * (err > 25).mean()), 1)],
    })
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(err, bins=50)
    ax.axvline(0, color="red", linewidth=1)
    ax.set_title(f"Prediction error with a {margin} kg safety margin ({model})")
    ax.set_xlabel("Predicted minus actual (kg)  —  right of the red line = overpredicted")
    ax.set_ylabel("Number of lifters")
    fig.tight_layout()
    return summary, fig


with gr.Blocks(title="Powerlifting Season Planner") as demo:
    gr.Markdown("# Powerlifting Season Planner\n"
                "Predicting a lifter's total at their next meet from their competition history "
                "(Spanish federation AEP, test period July 2025 – June 2026).")

    with gr.Tab("Model comparison"):
        gr.Dataframe(metrics, label="Test-set results")
        gr.Plot(plot_pred_vs_actual(), label="Predicted vs actual")
        gr.Dataframe(coefs.round(2).reset_index(names="feature"), label="Learned coefficients (standardized features)")

    with gr.Tab("Distributions"):
        column = gr.Dropdown(list(features.columns), value="TotalKg", label="Feature or target")
        dist_plot = gr.Plot(plot_distribution("TotalKg"))
        column.change(plot_distribution, inputs=column, outputs=dist_plot)

    with gr.Tab("Safety margin"):
        gr.Markdown("Overpredicting is the costly mistake: a lifter enters a meet expecting a total "
                    "they don't reach. Lowering every prediction by a safety margin trades overall "
                    "accuracy for fewer overpredictions.")
        with gr.Row():
            margin = gr.Slider(0, 50, value=0, step=1, label="Safety margin (kg)")
            model = gr.Dropdown(MODELS[1:], value="PyTorch nn.Module", label="Model")
        table = gr.Dataframe()
        err_plot = gr.Plot()
        margin.change(safety_margin, inputs=[margin, model], outputs=[table, err_plot])
        model.change(safety_margin, inputs=[margin, model], outputs=[table, err_plot])
        demo.load(safety_margin, inputs=[margin, model], outputs=[table, err_plot])

demo.launch()